In [1]:
import os
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set professional visualization style
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})

# Standard UCI Heart Disease Dataset Column Names
COLUMNS = [
    "age", "sex", "cp", "trestbps", "chol", "fbs",
    "restecg", "thalach", "exang", "oldpeak", "slope",
    "ca", "thal", "target"
]

def acquire_data(save_dir: str = "data/raw") -> pd.DataFrame:
    """
    Acquires the Heart Disease UCI dataset from a public source.
    Downloads the Cleveland dataset from the UCI Machine Learning Repository mirror.
    """
    os.makedirs(save_dir, exist_ok=True)
    file_path = os.path.join(save_dir, "heart.csv")

    # URL for Cleveland dataset from UCI repository mirror
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

    if not os.path.exists(file_path):
        print(f"[*] Downloading dataset from UCI Repository to {file_path}...")
        urllib.request.urlretrieve(url, file_path)
        print("[+] Download complete.")
    else:
        print(f"[*] Dataset already exists at {file_path}.")

    # Load dataset, converting '?' missing markers to NaN
    df = pd.read_csv(file_path, names=COLUMNS, na_values="?")
    return df

def clean_and_preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """
    Inspects data types, handles missing values, and formats the target variable.
    """
    print("\n" + "="*50)
    print("DATA INSPECTION & CLEANING")
    print("="*50)

    print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    print("\nData Types & Missing Values:")
    print(df.isna().sum())

    df_cleaned = df.copy()

    # Handle missing values in numerical columns using median imputation
    for col in df_cleaned.columns:
        if df_cleaned[col].isna().sum() > 0:
            median_val = df_cleaned[col].median()
            df_cleaned[col] = df_cleaned[col].fillna(median_val)
            print(f"[!] Imputed {col} missing values with median: {median_val}")

    # Standardize target to binary (0 = No Heart Disease, 1 = Heart Disease)
    # UCI raw data uses 0 for absence and 1, 2, 3, 4 for presence
    df_cleaned['target'] = (df_cleaned['target'] > 0).astype(int)

    return df_cleaned

def perform_eda(df: pd.DataFrame, output_dir: str = "reports/figures") -> None:
    """
    Generates professional EDA visualizations:
    1. Target Class Balance
    2. Feature Distribution Histograms
    3. Correlation Heatmap
    """
    os.makedirs(output_dir, exist_ok=True)
    print("\n" + "="*50)
    print("GENERATING EDA VISUALIZATIONS")
    print("="*50)

    # 1. Target Class Balance
    plt.figure(figsize=(7, 5))
    ax = sns.countplot(x='target', hue='target', data=df, palette=['#2ecc71', '#e74c3c'], legend=False)
    plt.title('Target Class Balance (Heart Disease Risk)', fontsize=14, fontweight='bold')
    plt.xlabel('Heart Disease (0 = No, 1 = Yes)', fontsize=12)
    plt.ylabel('Patient Count', fontsize=12)
    plt.xticks([0, 1], ['No Risk (0)', 'High Risk (1)'])

    # Annotate bar counts
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}',
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 5), textcoords='offset points')

    class_balance_path = os.path.join(output_dir, "class_balance.png")
    plt.savefig(class_balance_path, dpi=300)
    plt.close()
    print(f"[+] Saved Class Balance Chart: {class_balance_path}")

    # 2. Histograms for Continuous Features
    continuous_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
    plt.figure(figsize=(14, 10))
    for i, col in enumerate(continuous_features, 1):
        plt.subplot(2, 3, i)
        sns.histplot(df[col], kde=True, color='#3498db', bins=15)
        plt.title(f'Distribution of {col.capitalize()}', fontsize=12, fontweight='bold')
        plt.xlabel(col)
        plt.ylabel('Frequency')

    hist_path = os.path.join(output_dir, "feature_distributions.png")
    plt.savefig(hist_path, dpi=300)
    plt.close()
    print(f"[+] Saved Feature Distributions: {hist_path}")

    # 3. Correlation Heatmap
    plt.figure(figsize=(12, 9))
    corr = df.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap='coolwarm',
                vmin=-1, vmax=1, linewidths=0.5, cbar_kws={"shrink": .8})
    plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')

    heatmap_path = os.path.join(output_dir, "correlation_heatmap.png")
    plt.savefig(heatmap_path, dpi=300)
    plt.close()
    print(f"[+] Saved Correlation Heatmap: {heatmap_path}")

if __name__ == "__main__":
    # 1. Acquire Data
    raw_df = acquire_data()

    # 2. Clean Data
    cleaned_df = clean_and_preprocess(raw_df)

    # Save cleaned dataset for downstream training tasks
    os.makedirs("data/processed", exist_ok=True)
    cleaned_df.to_csv("data/processed/heart_cleaned.csv", index=False)
    print("\n[+] Saved cleaned dataset to 'data/processed/heart_cleaned.csv'")

    # 3. Perform EDA & Export Visualizations
    perform_eda(cleaned_df)
    print("\n[✔] Step 1 (Data Acquisition & EDA) completed successfully!")

[*] Downloading dataset from UCI Repository to data/raw/heart.csv...
[+] Download complete.

DATA INSPECTION & CLEANING
Dataset Shape: 303 rows, 14 columns

Data Types & Missing Values:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
target      0
dtype: int64
[!] Imputed ca missing values with median: 0.0
[!] Imputed thal missing values with median: 3.0

[+] Saved cleaned dataset to 'data/processed/heart_cleaned.csv'

GENERATING EDA VISUALIZATIONS
[+] Saved Class Balance Chart: reports/figures/class_balance.png
[+] Saved Feature Distributions: reports/figures/feature_distributions.png
[+] Saved Correlation Heatmap: reports/figures/correlation_heatmap.png

[✔] Step 1 (Data Acquisition & EDA) completed successfully!


In [2]:
import os
import joblib
import pandas as pd
import numpy as np
from typing import Dict, Any, Tuple

from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, roc_auc_score, classification_report
)

# Optional MLflow integration for Experiment Tracking (Task 3)
try:
    import mlflow
    import mlflow.sklearn
    MLFLOW_AVAILABLE = True
except ImportError:
    MLFLOW_AVAILABLE = False


def load_cleaned_data(file_path: str = "data/processed/heart_cleaned.csv") -> Tuple[pd.DataFrame, pd.Series]:
    """Loads cleaned heart dataset and splits into features (X) and target (y)."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Cleaned dataset not found at '{file_path}'. Please run Step 1 first.")

    df = pd.read_csv(file_path)
    X = df.drop(columns=['target'])
    y = df['target']
    return X, y


def build_preprocessor() -> Tuple[ColumnTransformer, list, list]:
    """
    Constructs a ColumnTransformer to scale continuous numerical features
    and one-hot encode categorical features without data leakage.
    """
    # Identify feature groupings based on UCI Heart Disease dataset definitions
    numerical_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
    categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

    # Pipelines for numerical and categorical features
    num_pipeline = Pipeline([
        ('scaler', StandardScaler())
    ])

    cat_pipeline = Pipeline([
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_pipeline, numerical_cols),
            ('cat', cat_pipeline, categorical_cols)
        ]
    )

    return preprocessor, numerical_cols, categorical_cols


def train_and_evaluate_models(X_train: pd.DataFrame, y_train: pd.Series, X_test: pd.DataFrame, y_test: pd.Series, preprocessor: ColumnTransformer) -> Dict[str, Any]:
    """
    Trains Logistic Regression and Random Forest models using Stratified K-Fold CV GridSearch,
    evaluates on test set, and logs metrics.
    """
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # 1. Logistic Regression Pipeline & GridSearch
    pipe_lr = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(random_state=42, max_iter=1000))
    ])

    param_grid_lr = {
        'classifier__C': [0.01, 0.1, 1.0, 10.0],
        'classifier__solver': ['lbfgs', 'liblinear']
    }

    # 2. Random Forest Pipeline & GridSearch
    pipe_rf = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42))
    ])

    param_grid_rf = {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__max_depth': [None, 5, 10],
        'classifier__min_samples_split': [3, 4]
    }

    models_config = {
        "Logistic_Regression": (pipe_lr, param_grid_lr),
        "Random_Forest": (pipe_rf, param_grid_rf)
    }

    results = {}
    best_overall_score = -1.0
    best_overall_model = None
    best_overall_name = ""

    print("\n" + "="*60)
    print("MODEL TRAINING & CROSS-VALIDATION HYPERPARAMETER TUNING")
    print("="*60)

    for name, (pipeline, param_grid) in models_config.items():
        print(f"\n[*] Tuning {name} with 5-Fold Stratified Cross-Validation...")
        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=cv,
            scoring='roc_auc',
            n_jobs=-1
        )
        grid_search.fit(X_train, y_train)

        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test)
        y_proba = best_model.predict_proba(X_test)[:, 1]

        # Calculate evaluation metrics
        metrics = {
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred),
            "Recall": recall_score(y_test, y_pred),
            "ROC-AUC": roc_auc_score(y_test, y_proba),
            "Best_CV_ROC_AUC": grid_search.best_score_,
            "Best_Params": grid_search.best_params_
        }

        results[name] = {
            "model": best_model,
            "metrics": metrics
        }

        print(f"[+] Best Params for {name}: {grid_search.best_params_}")
        print(f"    - CV ROC-AUC Score: {grid_search.best_score_:.4f}")
        print(f"    - Test Accuracy:   {metrics['Accuracy']:.4f}")
        print(f"    - Test Precision:  {metrics['Precision']:.4f}")
        print(f"    - Test Recall:     {metrics['Recall']:.4f}")
        print(f"    - Test ROC-AUC:    {metrics['ROC-AUC']:.4f}")

        # Track overall top performing model based on ROC-AUC
        if metrics["ROC-AUC"] > best_overall_score:
            best_overall_score = metrics["ROC-AUC"]
            best_overall_model = best_model
            best_overall_name = name

        # Optional MLflow experiment tracking logging
        if MLFLOW_AVAILABLE:
            with mlflow.start_run(run_name=name):
                mlflow.log_params(grid_search.best_params_)
                mlflow.log_metrics({
                    "test_accuracy": metrics["Accuracy"],
                    "test_precision": metrics["Precision"],
                    "test_recall": metrics["Recall"],
                    "test_roc_auc": metrics["ROC-AUC"],
                    "cv_roc_auc": grid_search.best_score_
                })
                # Trust necessary internal types to allow logging/loading with skops format
                mlflow.sklearn.log_model(
                    best_model,
                    artifact_path=f"model_{name}",
                    skops_trusted_types=[
                        "sklearn.compose._column_transformer._RemainderColsList",
                        "sklearn.tree._tree.Tree"
                    ]
                )

    print("\n" + "="*60)
    print(f"BEST MODEL SELECTED: {best_overall_name} (Test ROC-AUC: {best_overall_score:.4f})")
    print("="*60)

    return {
        "all_results": results,
        "best_model_name": best_overall_name,
        "best_model": best_overall_model
    }


def save_artifacts(best_model_pipeline: Pipeline, output_dir: str = "models") -> None:
    """Saves the final trained end-to-end pipeline to disk using joblib for downstream deployment."""
    os.makedirs(output_dir, exist_ok=True)

    # Save complete pipeline containing preprocessor + classifier
    pipeline_path = os.path.join(output_dir, "best_pipeline.joblib")
    joblib.dump(best_model_pipeline, pipeline_path)
    print(f"\n[+] Saved full model pipeline artifact: '{pipeline_path}'")


if __name__ == "__main__":
    # 1. Load Data
    X, y = load_cleaned_data()

    # 2. Train / Test Split (Stratified to maintain class ratio)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 3. Construct Preprocessor
    preprocessor, num_cols, cat_cols = build_preprocessor()

    # 4. Set MLflow Experiment (if installed)
    if MLFLOW_AVAILABLE:
        mlflow.set_experiment("Heart_Disease_MLOps_Assignment")

    # 5. Train, Tune, and Evaluate Models
    eval_results = train_and_evaluate_models(X_train, y_train, X_test, y_test, preprocessor)

    # 6. Save Best Model Pipeline for Packaging & Docker Container Serving (Task 4 & 6)
    save_artifacts(eval_results["best_model"])

    print("\n[✔] Step 2 (Feature Engineering & Model Development) completed successfully!")


MODEL TRAINING & CROSS-VALIDATION HYPERPARAMETER TUNING

[*] Tuning Logistic_Regression with 5-Fold Stratified Cross-Validation...
[+] Best Params for Logistic_Regression: {'classifier__C': 1.0, 'classifier__solver': 'lbfgs'}
    - CV ROC-AUC Score: 0.9025
    - Test Accuracy:   0.8852
    - Test Precision:  0.8387
    - Test Recall:     0.9286
    - Test ROC-AUC:    0.9665

[*] Tuning Random_Forest with 5-Fold Stratified Cross-Validation...
[+] Best Params for Random_Forest: {'classifier__max_depth': 10, 'classifier__min_samples_split': 4, 'classifier__n_estimators': 100}
    - CV ROC-AUC Score: 0.9075
    - Test Accuracy:   0.9016
    - Test Precision:  0.8438
    - Test Recall:     0.9643
    - Test ROC-AUC:    0.9426

BEST MODEL SELECTED: Logistic_Regression (Test ROC-AUC: 0.9665)

[+] Saved full model pipeline artifact: 'models/best_pipeline.joblib'

[✔] Step 2 (Feature Engineering & Model Development) completed successfully!


In [3]:
!pip install -q mlflow

import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, RocCurveDisplay
)
# Removed the step2 import since the functions are already defined globally in the notebook state
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline


def plot_and_log_artifacts(y_test, y_pred, y_proba, model_name: str, run_dir: str = "reports/figures"):
    """Generates and returns file paths for Confusion Matrix and ROC Curve plots."""
    os.makedirs(run_dir, exist_ok=True)

    # 1. Confusion Matrix Plot
    plt.figure(figsize=(6, 5))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Risk', 'Risk'], yticklabels=['No Risk', 'Risk'])
    plt.title(f'Confusion Matrix - {model_name}', fontsize=12, fontweight='bold')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    cm_path = os.path.join(run_dir, f"{model_name}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300, bbox_inches='tight')
    plt.close()

    # 2. ROC Curve Plot
    plt.figure(figsize=(6, 5))
    RocCurveDisplay.from_predictions(y_test, y_proba, name=model_name)
    plt.title(f'ROC Curve - {model_name}', fontsize=12, fontweight='bold')
    roc_path = os.path.join(run_dir, f"{model_name}_roc_curve.png")
    plt.savefig(roc_path, dpi=300, bbox_inches='tight')
    plt.close()

    return cm_path, roc_path


def run_experiment_tracking():
    """Runs MLflow tracking for Logistic Regression and Random Forest models."""
    # Set MLflow experiment name
    mlflow.set_experiment("Heart_Disease_Prediction_MLOps")

    # Load and split data (functions loaded globally from the previous notebook cell)
    X, y = load_cleaned_data()
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    preprocessor, _, _ = build_preprocessor()

    # Define candidate model configurations
    experiments = {
        "Logistic_Regression": {
            "model": LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=42),
            "params": {"C": 1.0, "solver": "lbfgs", "max_iter": 1000}
        },
        "Random_Forest": {
            "model": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
            "params": {"n_estimators": 100, "max_depth": 5, "min_samples_split": 2}
        }
    }

    print("\n" + "="*60)
    print("STEP 3: RUNNING MLFLOW EXPERIMENT TRACKING")
    print("="*60)

    for name, exp in experiments.items():
        with mlflow.start_run(run_name=name):
            # Construct complete pipeline
            pipeline = Pipeline([
                ('preprocessor', preprocessor),
                ('classifier', exp["model"])
            ])

            # Fit model
            pipeline.fit(X_train, y_train)

            # Predictions
            y_pred = pipeline.predict(X_test)
            y_proba = pipeline.predict_proba(X_test)[:, 1]

            # Calculate metrics
            acc = accuracy_score(y_test, y_pred)
            prec = precision_score(y_test, y_pred)
            rec = recall_score(y_test, y_pred)
            auc = roc_auc_score(y_test, y_proba)

            # 1. Log Hyperparameters
            mlflow.log_params(exp["params"])

            # 2. Log Metrics
            mlflow.log_metrics({
                "accuracy": acc,
                "precision": prec,
                "recall": rec,
                "roc_auc": auc
            })

            # 3. Generate & Log Plot Artifacts
            cm_path, roc_path = plot_and_log_artifacts(y_test, y_pred, y_proba, name)
            mlflow.log_artifact(cm_path, artifact_path="evaluation_plots")
            mlflow.log_artifact(roc_path, artifact_path="evaluation_plots")

            # 4. Log Model Artifact (Trust Column Transformer lists and Tree structures)
            mlflow.sklearn.log_model(
                pipeline,
                artifact_path="model",
                skops_trusted_types=[
                    "sklearn.compose._column_transformer._RemainderColsList",
                    "sklearn.tree._tree.Tree"
                ]
            )

            print(f"[+] Logged MLflow Run: {name}")
            print(f"    - Accuracy:  {acc:.4f} | Precision: {prec:.4f}")
            print(f"    - Recall:    {rec:.4f} | ROC-AUC:   {auc:.4f}")

    print("\n[✔] Step 3 complete. Launch MLflow UI via terminal: `mlflow ui`")


if __name__ == "__main__":
    run_experiment_tracking()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2026/10/13 13:03:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/13 13:03:23 INFO mlflow.store.db.utils: Updating database tables
2026/10/13 13:03:26 INFO mlflow.tracking.fluent: Experiment with name 'Heart_Disease_Prediction_MLOps' does not exist. Creating a new experiment.



STEP 3: RUNNING MLFLOW EXPERIMENT TRACKING


2026/10/13 13:03:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[+] Logged MLflow Run: Logistic_Regression
    - Accuracy:  0.8852 | Precision: 0.8387
    - Recall:    0.9286 | ROC-AUC:   0.9665


2026/10/13 13:03:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[+] Logged MLflow Run: Random_Forest
    - Accuracy:  0.8689 | Precision: 0.8333
    - Recall:    0.8929 | ROC-AUC:   0.9481

[✔] Step 3 complete. Launch MLflow UI via terminal: `mlflow ui`


<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

In [4]:
import os
import joblib
import pandas as pd
import numpy as np

def package_and_verify():
    """
    Verifies that the serialized preprocessing + model pipeline artifact
    loads cleanly and produces reproducible predictions.
    """
    model_path = "models/best_pipeline.joblib"

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model package '{model_path}' not found.")

    print("\n" + "="*60)
    print("STEP 4: MODEL PACKAGING & REPRODUCIBILITY VERIFICATION")
    print("="*60)

    # 1. Load serialized model pipeline artifact
    pipeline = joblib.load(model_path)
    print(f"[+] Successfully loaded serialized pipeline from '{model_path}'")

    # 2. Construct sample test payload matching standard UCI raw schema
    sample_patient = pd.DataFrame([{
        "age": 57,
        "sex": 1,
        "cp": 3,
        "trestbps": 145,
        "chol": 233,
        "fbs": 1,
        "restecg": 0,
        "thalach": 150,
        "exang": 0,
        "oldpeak": 2.3,
        "slope": 0,
        "ca": 0,
        "thal": 1
    }])

    # 3. Test end-to-end inference reproducibility across separate calls
    prediction_1 = pipeline.predict(sample_patient)
    proba_1 = pipeline.predict_proba(sample_patient)[0, 1]

    prediction_2 = pipeline.predict(sample_patient)
    proba_2 = pipeline.predict_proba(sample_patient)[0, 1]

    # Check consistency
    assert prediction_1 == prediction_2, "Inconsistent model predictions detected!"
    assert np.isclose(proba_1, proba_2), "Inconsistent confidence scores detected!"

    print(f"[+] Sample Patient Raw Features:\n{sample_patient.to_dict(orient='records')}")
    print(f"[+] Predicted Class: {prediction_1} (0 = No Risk, 1 = High Risk)")
    print(f"[+] Risk Probability (Confidence): {proba_1:.4f}")
    print("[✔] Reproducibility test passed: Predictions are identical across runs.")

if __name__ == "__main__":
    package_and_verify()


STEP 4: MODEL PACKAGING & REPRODUCIBILITY VERIFICATION
[+] Successfully loaded serialized pipeline from 'models/best_pipeline.joblib'
[+] Sample Patient Raw Features:
[{'age': 57, 'sex': 1, 'cp': 3, 'trestbps': 145, 'chol': 233, 'fbs': 1, 'restecg': 0, 'thalach': 150, 'exang': 0, 'oldpeak': 2.3, 'slope': 0, 'ca': 0, 'thal': 1}]
[+] Predicted Class: [0] (0 = No Risk, 1 = High Risk)
[+] Risk Probability (Confidence): 0.1917
[✔] Reproducibility test passed: Predictions are identical across runs.


In [5]:
import os
import pytest
import pandas as pd
import numpy as np

# Removed external script imports since these functions are already active in the notebook's global state
# from step1_data_acquisition_eda import clean_and_preprocess
# from step2_feature_engineering_model_training import build_preprocessor, load_cleaned_data

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


@pytest.fixture
def sample_raw_data():
    """Fixture providing a synthetic raw dataframe with missing values."""
    return pd.DataFrame({
        "age": [63.0, 67.0, 67.0, 37.0, np.nan],
        "sex": [1.0, 1.0, 1.0, 1.0, 0.0],
        "cp": [1.0, 4.0, 4.0, 3.0, 2.0],
        "trestbps": [145.0, 160.0, 120.0, 130.0, 130.0],
        "chol": [233.0, 286.0, 229.0, 250.0, 236.0],
        "fbs": [1.0, 0.0, 0.0, 0.0, 0.0],
        "restecg": [2.0, 2.0, 2.0, 0.0, 0.0],
        "thalach": [150.0, 108.0, 129.0, 187.0, 178.0],
        "exang": [0.0, 1.0, 1.0, 0.0, 0.0],
        "oldpeak": [2.3, 1.5, 2.6, 3.5, 1.4],
        "slope": [3.0, 2.0, 2.0, 3.0, 2.0],
        "ca": [0.0, 3.0, 2.0, 0.0, np.nan],
        "thal": [6.0, 3.0, 7.0, 3.0, 3.0],
        "target": [2, 4, 1, 0, 1]  # Raw target includes values > 1
    })


def test_clean_and_preprocess_handles_missing_and_target(sample_raw_data):
    """Test that missing values are imputed and target is converted to binary."""
    df_cleaned = clean_and_preprocess(sample_raw_data)

    # 1. Verify missing values are imputed
    assert df_cleaned.isna().sum().sum() == 0, "Missing values were not properly imputed!"

    # 2. Verify target is strictly binary (0 or 1)
    unique_targets = set(df_cleaned['target'].unique())
    assert unique_targets.issubset({0, 1}), f"Unexpected target values found: {unique_targets}"


def test_preprocessor_transformation(sample_raw_data):
    """Test that ColumnTransformer correctly processes numerical and categorical features."""
    df_cleaned = clean_and_preprocess(sample_raw_data)
    X_feat = df_cleaned.drop(columns=['target'])

    preprocessor, num_cols, cat_cols = build_preprocessor()
    X_transformed = preprocessor.fit_transform(X_feat)

    # Check output shape
    assert X_transformed.shape[0] == len(df_cleaned)
    assert X_transformed.shape[1] > 0
    assert not np.isnan(X_transformed).any(), "Transformed feature matrix contains NaNs!"


def test_full_pipeline_prediction_shape_and_range():
    """Test that the trained Scikit-Learn pipeline yields expected prediction formats."""
    X_data, y_data = load_cleaned_data()
    preprocessor, _, _ = build_preprocessor()

    # Build simple test pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(random_state=42))
    ])
    pipeline.fit(X_data, y_data)

    # Sample single instance prediction
    sample_input = X_data.iloc[:5]
    predictions = pipeline.predict(sample_input)
    probabilities = pipeline.predict_proba(sample_input)

    assert len(predictions) == 5
    assert set(predictions).issubset({0, 1})
    assert probabilities.shape == (5, 2)
    # Ensure probabilities sum to 1
    np.testing.assert_allclose(probabilities.sum(axis=1), 1.0, rtol=1e-5)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [6]:
import os
import time
import logging
import joblib
import pandas as pd
from typing import Dict, Any

from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel, Field

# Configure logging for Step 8 monitoring requirements
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("heart_disease_api")

app = FastAPI(
    title="Heart Disease Risk Prediction API",
    description="MLOps Model Serving Endpoint for Patient Heart Disease Risk Classification",
    version="1.0.0"
)

MODEL_PATH = "models/best_pipeline.joblib"
model_pipeline = None


# Load model pipeline at application startup
@app.on_event("startup")
def load_model():
    global model_pipeline
    if not os.path.exists(MODEL_PATH):
        logger.error(f"Model file not found at {MODEL_PATH}")
        raise RuntimeError(f"Model file not found at '{MODEL_PATH}'. Ensure step2/4 completed successfully.")

    model_pipeline = joblib.load(MODEL_PATH)
    logger.info(f"Successfully loaded model pipeline from {MODEL_PATH}")


# Middleware for request logging and latency tracking
@app.middleware("http")
def log_requests(request: Request, call_next):
    start_time = time.time()
    response = call_next(request)
    duration = time.time() - start_time
    logger.info(f"Path: {request.url.path} | Method: {request.method} | Status: {response.status_code} | Duration: {duration:.4f}s")
    return response


# Pydantic schema enforcing standard UCI Heart Disease feature types
class PatientData(BaseModel):
    age: int = Field(..., example=57, description="Age in years")
    sex: int = Field(..., example=1, description="Sex (1 = male; 0 = female)")
    cp: int = Field(..., example=3, description="Chest pain type (0-3)")
    trestbps: int = Field(..., example=145, description="Resting blood pressure (mm Hg)")
    chol: int = Field(..., example=233, description="Serum cholesterol in mg/dl")
    fbs: int = Field(..., example=1, description="Fasting blood sugar > 120 mg/dl (1 = true; 0 = false)")
    restecg: int = Field(..., example=0, description="Resting ECG results (0-2)")
    thalach: int = Field(..., example=150, description="Maximum heart rate achieved")
    exang: int = Field(..., example=0, description="Exercise induced angina (1 = yes; 0 = no)")
    oldpeak: float = Field(..., example=2.3, description="ST depression induced by exercise")
    slope: int = Field(..., example=0, description="Slope of peak exercise ST segment (0-2)")
    ca: int = Field(..., example=0, description="Number of major vessels (0-3) colored by flourosopy")
    thal: int = Field(..., example=1, description="Thalassemia type (1 = normal; 2 = fixed defect; 3 = reversable defect)")


class PredictionResponse(BaseModel):
    prediction: int = Field(..., description="0 = No Risk, 1 = High Risk")
    risk_label: str = Field(..., description="Human-readable prediction label")
    confidence: float = Field(..., description="Probability confidence score for predicted class")
    probabilities: Dict[str, float] = Field(..., description="Probabilities for both classes")


@app.get("/")
def read_root():
    """Health check endpoint."""
    return {"status": "healthy", "service": "Heart Disease Risk Prediction API", "model_loaded": model_pipeline is not None}


@app.get("/health")
def health_check():
    """Liveness probe for Kubernetes / container monitoring."""
    if model_pipeline is None:
        raise HTTPException(status_code=503, detail="Model artifact is not loaded.")
    return {"status": "UP"}


@app.post("/predict", response_model=PredictionResponse)
def predict_heart_disease(patient: PatientData):
    """
    Accepts JSON patient health record and returns class prediction + confidence score.
    """
    if model_pipeline is None:
        raise HTTPException(status_code=500, detail="Model pipeline is not initialized.")

    try:
        # Convert input Pydantic model to DataFrame matching training schema
        input_df = pd.DataFrame([patient.dict()])

        # Inference
        prediction = int(model_pipeline.predict(input_df))
        probabilities = model_pipeline.predict_proba(input_df)

        confidence = float(probabilities[prediction])
        label = "High Risk" if prediction == 1 else "No Risk"

        return PredictionResponse(
            prediction=prediction,
            risk_label=label,
            confidence=round(confidence, 4),
            probabilities={
                "no_risk": round(float(probabilities), 4),
                "high_risk": round(float(probabilities[3]), 4)
            }
        )

    except Exception as e:
        logger.error(f"Inference error: {str(e)}")
        raise HTTPException(status_code=400, detail=f"Prediction failed: {str(e)}")

/tmp/ipykernel_1668/1665895660.py:29: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")
/tmp/ipykernel_1668/1665895660.py:52: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  age: int = Field(..., example=57, description="Age in years")
/tmp/ipykernel_1668/1665895660.py:53: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13

In [7]:
import os

workflow_dir = '.github/workflows'
exists = os.path.exists(workflow_dir)

print('='*60)
print('VERIFYING GITHUB WORKFLOWS DIRECTORY')
print('='*60)
print(f'[?] Path \'{workflow_dir}\' exists: {exists}')

if exists:
    files = os.listdir(workflow_dir)
    print(f'[+] Files found: {files}')
else:
    print('[-] The folder was not created in the local file system. Creating it now...')
    os.makedirs(workflow_dir, exist_ok=True)
    print('[+] Empty .github/workflows/ directory successfully created.')

VERIFYING GITHUB WORKFLOWS DIRECTORY
[?] Path '.github/workflows' exists: False
[-] The folder was not created in the local file system. Creating it now...
[+] Empty .github/workflows/ directory successfully created.


In [8]:
workflow_content = """name: MLOps CI/CD Pipeline

on:
  push:
    branches: [ main, master ]
  pull_request:
    branches: [ main, master ]

jobs:
  continuous-integration:
    runs-on: ubuntu-latest
    steps:
    - name: Checkout Code
      uses: actions/checkout@v3

    - name: Set up Python
      uses: actions/setup-python@v4
      with:
        python-version: '3.10'
        cache: 'pip'

    - name: Install Dependencies
      run: |
        python -m pip install --upgrade pip
        pip install pytest pandas numpy scikit-learn joblib fastapi pydantic uvicorn httpx

    - name: Run Unit Tests
      run: |
        pytest -v

    - name: Verify Model Artifacts
      run: |
        python -c "
        import os, joblib
        assert os.path.exists('models/best_pipeline.joblib'), 'Model artifact not found!'
        pipeline = joblib.load('models/best_pipeline.joblib')
        print('[+] Model artifact successfully verified in CI workflow.')
        "
"""

workflow_file_path = os.path.join(workflow_dir, 'mlops-ci-cd.yml')
with open(workflow_file_path, 'w') as f:
    f.write(workflow_content)

print('='*60)
print('WRITING GITHUB ACTIONS WORKFLOW FILE')
print('='*60)
print(f'[+] Successfully wrote CI/CD pipeline config to: {workflow_file_path}')


WRITING GITHUB ACTIONS WORKFLOW FILE
[+] Successfully wrote CI/CD pipeline config to: .github/workflows/mlops-ci-cd.yml
